# Day 2 — Data Cleaning

This notebook demonstrates cleaning steps for the Day 2 datasets: parsing dates, handling missing values, de-duplicating, encoding categorical fields, scaling numeric features, and saving cleaned outputs to `data/processed/`.

Follow the cells below to run individual steps interactively.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import create_engine

pd.set_option('display.max_columns', 50)

# Paths
ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
DB_DIR = ROOT / 'data' / 'db'
DB_PATH = DB_DIR / 'bluestock_mf.db'

DATE_COLUMNS = {
    '02_nav_history.csv': ['date'],
    '08_investor_transactions.csv': ['transaction_date'],
    '07_scheme_performance.csv': [],
}

TRANSACTION_TYPE_MAP = {
    'sip': 'SIP', 'sips': 'SIP', 'lumpsum': 'Lumpsum', 'lump sum': 'Lumpsum',
    'lump-sum': 'Lumpsum', 'redemption': 'Redemption', 'redeem': 'Redemption'
}

ALLOWED_KYC_STATUS = {'Verified', 'Pending', 'Rejected', 'Not Submitted'}


In [ ]:
# Load a sample dataset (NAV history)
nav_path = RAW_DIR / '02_nav_history.csv'
df_nav = pd.read_csv(nav_path, nrows=200)
print('shape:', df_nav.shape)
df_nav.head()


In [ ]:
# Inspect data structure
print(df_nav.dtypes)
print('\nMissing values summary:\n', df_nav.isna().sum())
print('\nDuplicates:', int(df_nav.duplicated().sum()))


In [ ]:
# Cleaning helpers

def parse_date_column(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors='coerce').dt.date


def clean_nav_history(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'date' in df.columns:
        df['date'] = parse_date_column(df['date'])
    if 'nav' in df.columns:
        df['nav'] = pd.to_numeric(df['nav'], errors='coerce')
    df = df.sort_values(['amfi_code', 'date'])
    df = df.drop_duplicates(subset=['amfi_code', 'date'], keep='last')

    filled = []
    for amfi, g in df.groupby('amfi_code'):
        g = g.set_index('date').sort_index()
        all_dates = pd.date_range(g.index.min(), g.index.max(), freq='D')
        g = g.reindex(all_dates)
        g['amfi_code'] = amfi
        g['nav'] = g['nav'].ffill()
        filled.append(g.reset_index().rename(columns={'index':'date'}))
    res = pd.concat(filled, ignore_index=True)
    res = res.dropna(subset=['date'])
    res = res[res['nav']>0]
    return res


def clean_investor_transactions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if 'transaction_date' in df.columns:
        df['transaction_date'] = parse_date_column(df['transaction_date'])
    if 'amount_inr' in df.columns:
        df['amount_inr'] = pd.to_numeric(df['amount_inr'], errors='coerce')
    if 'transaction_type' in df.columns:
        df['transaction_type'] = (
            df['transaction_type'].astype(str).str.strip().str.lower().replace(TRANSACTION_TYPE_MAP)
        )
        df['transaction_type'] = df['transaction_type'].map(lambda x: x if x in TRANSACTION_TYPE_MAP.values() else x.title())
    if 'kyc_status' in df.columns:
        df['kyc_status'] = df['kyc_status'].astype(str).str.strip().str.title()
        df.loc[~df['kyc_status'].isin(ALLOWED_KYC_STATUS), 'kyc_status'] = 'Unknown'
    df = df[df['amount_inr']>0]
    return df


def clean_scheme_performance(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    num_cols = [c for c in df.columns if df[c].dtype != object]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df


In [ ]:
# Before / After for NAV cleaning
raw_nav = pd.read_csv(RAW_DIR / '02_nav_history.csv')
print('raw rows:', raw_nav.shape[0])
cleaned_nav = clean_nav_history(raw_nav)
print('cleaned rows:', cleaned_nav.shape[0])
cleaned_nav.head()


In [ ]:
# Transactions cleaning example
raw_tx = pd.read_csv(RAW_DIR / '08_investor_transactions.csv')
print('raw tx rows:', raw_tx.shape[0])
clean_tx = clean_investor_transactions(raw_tx)
print('clean tx rows:', clean_tx.shape[0])
clean_tx.head()


In [ ]:
# Remove duplicates example
print('duplicates in transactions:', int(raw_tx.duplicated().sum()))
raw_tx_dedup = raw_tx.drop_duplicates()
print('after dedupe:', raw_tx_dedup.shape[0])


In [ ]:
# Encode categorical variables (example)
if 'transaction_type' in clean_tx.columns:
    print(clean_tx['transaction_type'].value_counts())
    dummies = pd.get_dummies(clean_tx['transaction_type'], prefix='tx')
    clean_tx = pd.concat([clean_tx, dummies], axis=1)
    print('one-hot columns added:', [c for c in clean_tx.columns if c.startswith('tx_')][:5])


In [ ]:
# Normalize numeric columns (example)
num_cols = ['amount_inr'] if 'amount_inr' in clean_tx.columns else []
for c in num_cols:
    clean_tx[f'{c}_z'] = (clean_tx[c] - clean_tx[c].mean()) / clean_tx[c].std()
print('Added z-score columns:', [c for c in clean_tx.columns if c.endswith('_z')])


In [ ]:
# Save cleaned datasets
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
cleaned_nav.to_csv(PROCESSED_DIR / '02_nav_history_processed.csv', index=False)
clean_tx.to_csv(PROCESSED_DIR / '08_investor_transactions_processed.csv', index=False)
print('Wrote processed files to', PROCESSED_DIR)


# Quick example: load a cleaned table into SQLite
from sqlalchemy import create_engine
engine = create_engine(f'sqlite:///{DB_PATH}')
# example: write dim_fund (if available)
# fund_master = pd.read_csv(RAW_DIR / '01_fund_master.csv')
# fund_master.to_sql('dim_fund', engine, if_exists='replace', index=False)

# Note: For running the DDL in `sql/schema.sql` which contains multiple statements,
# use `engine.raw_connection().executescript(schema_sql)` rather than `conn.execute(text(schema_sql))`.


## Conclusion

This notebook provides interactive examples for Day 2 cleaning. Next steps:
- Expand cleaning functions to cover all datasets
- Add unit tests / assertions for data validation
- Integrate cells into `scripts/etl_pipeline.py` calls for automation
